# 第2回 演習：段を1本の式に。そして、本当に当たるか


---


## 前回からの接続


第1回では、気温帯・年・天気で分けた平均で「明日何台か」に答えました。毎日平均と答えると外れ幅の平均は1582台、気温帯×年で分けると762台でした。ただ、その答えは気温帯の境目で跳ぶガタガタの段でした。しかも、細かく分けるほど外れ幅は縮みましたが、それは平均を作るのに使った日で採点していたからでした。

今日は、この段を1本の式にします。そして、その式がまだ来ていない日にも当たるのかを確かめます。


---


## 今日の分析目標


気温を入れると台数が出てくる1本の式を立て、その式がまだ来ていない日にどれだけ当たるかを測る。

1日ぶんの計算から始めて、731日ぶん、総当たり、式一発と積み上げて「良い直線」を選びます。そのあと、データを分けて本当の実力を測ります。最後の「目標に答えられたか」で振り返りましょう。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 1日ぶんの予測・外れ・外れの二乗を計算し、`for`文で731日ぶん足し上げられる
- `for`文と`if`文の総当たりで二乗和の谷底を探し、最小二乗法（least squares）の考え方を説明できる
- 谷底を式一発で求め、`LinearRegression`の答えと一致することを確かめられる
- R²を「毎日平均と答える基準から、外れの二乗和をどれだけ減らせたか」として計算できる
- データを訓練用とテスト用に分け、作るのに使ったデータでの成績と、まだ見ていないデータでの成績を比べられる
- 式を曲げすぎたときの過学習（overfitting）と、交差検証（cross-validation）で分け方の運をならす考え方を説明できる


---


## 0. 設定


### 最初に実行する：道具を呼び出す設定のセル

- 役割：このノートで使う道具（ライブラリ）を呼び出し、データを読み込む
- 入力：なし
- 出力：「セットアップ完了」の表示と、表`df`

今日は、表のpandasと図のmatplotlibに加えて、数の並びをまとめて計算するnumpyと、回帰の道具が入ったscikit-learn（`sklearn`）を使います。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
try:
    import japanize_matplotlib
except Exception:   # japanize が動かなくなったときの保険：同梱フォントを直接登録する
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None) if spec else None
    if ttf:
        fm.fontManager.addfont(str(ttf))
        plt.rcParams['font.family'] = fm.FontProperties(fname=str(ttf)).get_name()

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ

# 自転車シェアの日次データ（1日1行）を読み込む
# データはこのリポジトリから読み込む
DATA_DIR = ('https://raw.githubusercontent.com/k0heiun0/'
            'applied_exercise_preview/main/shared/data')
df = pd.read_csv(f'{DATA_DIR}/bike_day.csv')
print('セットアップ完了')


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は ② の解析が中心です。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for i, (x0, p, c) in enumerate(zip([0.17, 0.5, 0.83], phases, colors), start=1):
    on = i == 2
    ax.text(x0, 0.62, p, ha='center', va='center', fontsize=13 if on else 11, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95 if on else 0.3))
for a0, a1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(a1, 0.62), xytext=(a0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


---


## 1. 列を、数の並びとして取り出す


今日は、前回の段を1本の式に置き換えます。式を立てるには、気温と台数を「1日ずつの数の並び」として扱えると便利です。


### おさらい：列を選ぶ

表から1つの列を取り出すには、`[ ]`に列名を入れます。

```python
df['cnt']        # 台数の列だけ
```

取り出したものは、日付の番号つきの列（Series）です。


### お手本：列を配列にして、1日目を見る

- 役割：気温と台数の列を、数の並び（配列）として取り出し、1日目の値を見る
- 入力：表`df`
- 出力：配列`x`（気温）と`y`（台数）

`.values`を付けると、列がnumpyの配列（array）になります。配列は`x[0]`のように番号で1つずつ取り出せます。番号は0から数えます。


In [ ]:
x = df['temp'].values     # 気温（731日ぶん）
y = df['cnt'].values      # 台数（731日ぶん）
print(len(x))             # いくつ並んでいるか
print(x[0])               # 1日目の気温
print(y[0])               # 1日目の台数


731個の数が並んでいます。1日目は、気温0.344（41を掛けると約14.1℃）で、台数は985台でした。


### 変えてみる：2日目を見る

`[0]`を`[1]`に変えれば、2日目です。


In [ ]:
print(x[1])               # 2日目の気温
print(y[1])               # 2日目の台数


---


## 2. 1日目を、直線で予測すると？


1節で、1日目の気温と台数を取り出せました。前回の段の代わりに、いちばん単純な式を考えます。直線です。

$$
\text{台数} \approx a \times \text{temp} + b
$$

`temp`は、気温（℃）を41で割って0〜1に縮めた値です。a（傾き）は、`temp`が1増えたときに台数がいくら増えるか。`temp`の1は約41℃なので、1℃あたりの増え方はa ÷ 41です（a=6600なら1℃で約160台）。b（切片）は、`temp`が0のときの台数です。aとbの2つの数を決めれば、式が完成します。でも、aとbの組み合わせは無数にあります。どれが「良い直線」なのでしょうか。まずは適当なaとbで、1日目だけを予測してみます。


### お手本：1日目の予測と外れ

- 役割：a=9000・b=0の直線で1日目を予測し、外れ（実際 − 予測）とその二乗を出す
- 入力：`x[0]`・`y[0]`と、a・b
- 出力：予測・外れ・外れの二乗

予測が実際からどれだけずれたかを残差（residual）と呼びます。外れは多すぎても少なすぎても困るので、符号を消すために二乗します。二乗にはもう1つ良さがあります。二乗の和は、aやbを動かすとなめらかな谷の形になるので、あとで見るように谷底を式一発で求められるのです（4節・5節）。大きく外れた日ほど重く数えられる、という性質もあります。


In [ ]:
a = 9000
b = 0
pred = a * x[0] + b       # 1日目の予測
print(pred)
err = y[0] - pred         # 外れ（実際 − 予測）
print(err)
print(err ** 2)           # 外れの二乗


予測は約3098台。実際は985台なので、外れは −2112.5台（多く見積もりすぎた）、その二乗は約446万です。


### TODO①：2日目と3日目も、同じように計算する

上のお手本のセルをコピーして、`x[0]`・`y[0]`を2日目（`[1]`）と3日目（`[2]`）に変え、予測・外れ・外れの二乗を表示してください。


In [ ]:
# TODO: 上のセルをコピーして、[0] を [1]（2日目）と
#       [2]（3日目）に変えて表示する
# ヒント: 2日目の分と3日目の分を、続けて書けばよい
...


2日目の外れは −2470.3、3日目は −418.3です。同じことを731日ぶん書くのは大変です。次の節で、繰り返しを書く方法を使います。


---


## 3. 731日ぶんを、足し上げる


直線の良さを測るには、1日だけでなく、全部の日の外れを見る必要があります。外れの二乗を全日で足した二乗和を、直線の物差しにします。小さいほど、よく当たる直線です。


### お手本：for文で足し上げる

- 役割：731日ぶんの外れの二乗を、1日ずつ足し上げる
- 入力：配列`x`・`y`と、a・b
- 出力：二乗和`total`

`for i in range(len(x)):`は、「`i`を0, 1, 2, …, 730と変えながら、下の字下げした行を繰り返す」という書き方です（for文）。字下げ（行の頭のすきま）が、繰り返す範囲の目印です。`total = total + …`は、「いまの合計に足して、合計を更新する」という意味です。


In [ ]:
total = 0
for i in range(len(x)):
    pred = a * x[i] + b                  # その日の予測
    total = total + (y[i] - pred) ** 2   # 外れの二乗を足す
print(total)


二乗和は約18.0億です。2節で計算した1日目の446万は、この中の1日ぶんです。


### 変えてみる：numpyで、まとめて計算する

numpyの配列は、`for`文を書かなくてもまとめて計算できます。`a * x + b`と書くと、731日ぶんの予測が一度に出ます。`.sum()`は全部を足すメソッドです。

`residual[:3]`は、配列の番号0・1・2の3つを取り出す書き方です（スライス）。`[始め:終わり]`の形で、始めを省くと0から、終わりの番号は含みません。


In [ ]:
residual = y - (a * x + b)       # 731日ぶんの外れを、まとめて
print(residual[:3])              # 最初の3日ぶん
print((residual ** 2).sum())     # 二乗して、全部足す


最初の3日ぶんは、2節とTODO①で1日ずつ計算した値と同じです。二乗和も`for`文の答えと一致しました。`for`文で書いたことを、numpyは1行でやってくれていたのです。


---


## 4. いちばん良いaとbは？　総当たりで探す


二乗和がいちばん小さくなるaとbを探します。まずは素朴に、aをいろいろ試してみます。


### お手本：aを100ずつ試す

- 役割：bを1200に固定し、aを5000〜8000まで100ずつ変えて、二乗和がいちばん小さいaを探す
- 入力：配列`x`・`y`
- 出力：いちばん良いaと、そのときの二乗和

`range(5000, 8001, 100)`のように数を3つ渡すと、`range(始め, 終わり, 刻み)`の意味になり、始めから刻みずつ増えていきます。終わりの数は含まないので、8000まで入れたいときは8001と書きます。まず、小さな範囲で確かめてみましょう。


In [ ]:
for a in range(5000, 5301, 100):
    print(a)          # 5000, 5100, 5200, 5300 と表示される


`if`文は「条件が成り立つときだけ、下の字下げした行を実行する」書き方です（if文）。ここでは「これまでの最小より小さければ、覚え直す」に使います。`None`は「まだ何も入っていない」という印で、1回目は必ず覚えます。`A or B`は「AとBのどちらかが成り立てば成り立つ」で、1回目は`best_sse is None`で成り立ち、2回目からは`s < best_sse`で決まります。


In [ ]:
best_a = None
best_sse = None
for a in range(5000, 8001, 100):              # 5000, 5100, …, 8000
    s = ((y - (a * x + 1200)) ** 2).sum()     # この a の二乗和
    if best_sse is None or s < best_sse:      # 小さければ覚え直す
        best_a = a
        best_sse = s
print(best_a, best_sse)


a=6700のとき、二乗和が最小になりました。aを動かしたときの二乗和を図にすると、谷底が1つだけのU字になります。


In [ ]:
# @title 図：a を動かしたときの二乗和
a_list = list(range(5000, 8001, 100))
sse_list = [((y - (a * x + 1200)) ** 2).sum() for a in a_list]
plt.plot(a_list, sse_list, 'o-', color='#0066cc')
plt.axvline(best_a, color='#e63946', ls='--')
plt.xlabel('傾き a（b は 1200 に固定）'); plt.ylabel('二乗和'); plt.show()


### 変えてみる：bも動かす（for文を二重に）

上のセルの`1200`を、もう1つの`for`文の`b`に変えます。`for`文の中に`for`文を入れると、aとbのすべての組み合わせを試せます。`best`には (a, b, 二乗和) の3つをまとめて覚えます。`best[2]`は3つ目、つまり二乗和です。


In [ ]:
best = None
for a in range(5000, 8001, 100):
    for b in range(0, 2001, 100):          # b も動かす
        s = ((y - (a * x + b)) ** 2).sum()
        if best is None or s < best[2]:
            best = (a, b, s)
print(best)


31×21 ＝ 651通りを試して、a=6700・b=1200が選ばれました。


### TODO②：範囲を狭めて、刻みを細かくする

上の二重の`for`文のセルをコピーして、aを6400〜6800、bを1000〜1400の範囲で、どちらも10ずつ試すように変えてください。


In [ ]:
# TODO: 上のセルをコピーして、a を range(6400, 6801, 10)、
#       b を range(1000, 1401, 10) にする
# ヒント: 変えるのは range の中の数だけ
...


---


## 5. 谷底を、式一発で求める


総当たりは、刻みを細かくするほど時間がかかります。実は、二乗和の谷底は式一発で求められます。二乗和をaとbで微分して0とおくと、次の式が出てきます（$\bar{x}$・$\bar{y}$ は平均）。

$$
a = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sum (x_i - \bar{x})^2}, \qquad b = \bar{y} - a\,\bar{x}
$$

外れの二乗和を最小にしてaとbを決めるこのやり方を、最小二乗法と呼びます。


### お手本：式をfor文で計算する

- 役割：上の式の分子と分母を`for`文で足し上げ、谷底のaとbを出す
- 入力：配列`x`・`y`
- 出力：谷底の`a_best`・`b_best`


In [ ]:
x_mean = x.mean()
y_mean = y.mean()
num = 0     # 分子
den = 0     # 分母
for i in range(len(x)):
    num = num + (x[i] - x_mean) * (y[i] - y_mean)
    den = den + (x[i] - x_mean) ** 2
a_best = num / den
b_best = y_mean - a_best * x_mean
print(a_best, b_best)


a=6640.71、b=1214.64。TODO②の総当たりの谷底（6650, 1210）のすぐ近くです。


### 答え合わせ：LinearRegressionと比べる

scikit-learnの`LinearRegression`は、この最小二乗法を中でやってくれる道具です。`fit`で学ばせ、`coef_`（傾き）と`intercept_`（切片）で結果を見ます。列は`df[['temp']]`のように二重のかっこ（表の形）で渡します。


In [ ]:
model_temp = LinearRegression()
model_temp.fit(df[['temp']], y)
print(model_temp.coef_[0])           # 傾き a
print(model_temp.intercept_)         # 切片 b


式一発の答えと一致しました。`fit()`の一行の中身は、今日ここまでやってきた計算そのものです。


### 変えてみる：R²を自分で計算する

当たり具合の物差しとして、R²（決定係数）を使います。「毎日平均と答えたときの二乗和（基準）のうち、直線でどれだけ減らせたか」の割合で、1に近いほどよく当たります。


In [ ]:
pred_line = a_best * x + b_best
sse_line = ((y - pred_line) ** 2).sum()      # 直線の二乗和
sst = ((y - y.mean()) ** 2).sum()            # 基準の二乗和
print(1 - sse_line / sst)                    # R²
print(model_temp.score(df[['temp']], y))     # 答え合わせ


R²は0.394。基準の二乗和のうち、約4割を気温の直線で減らせました。

実は、前回の気温帯の段のR²は0.439で、直線より少し上です。散布図で見たとおり、台数は暑すぎると頭打ちになるので、まっすぐな直線では右端を外してしまうのです。それでも直線には、境目で跳ばない、1本の式で書ける、という良さがあります。

手がかりを増やすこともできます。`df[['temp', 'yr']]`のように列を並べるだけで、気温と年を両方使う式（重回帰、multiple regression）になります。


In [ ]:
model_ty = LinearRegression()
model_ty.fit(df[['temp', 'yr']], y)
print(model_ty.coef_, model_ty.intercept_)
print(model_ty.score(df[['temp', 'yr']], y))


気温と年を使うと、R²は0.68まで上がりました。`coef_`に並ぶ、各列に掛かる数（直線の傾きaにあたる数）を、係数と呼びます。


---


## 6. 自己採点は、甘くないか


ここまでのR²は、式を作るのに使った731日で測っていました。第1回の最後で見たとおり、作るのに使った日で採点すると、成績は甘く出ます。本当の実力は、まだ見ていない日で測る必要があります。

そこで、データを2つに分けます。式を作るのに使う訓練データ（過去問）と、最後まで取っておくテストデータ（本番）です。まず、第1回の段で試してみます。


In [ ]:
# @title 第1回の段を、作った日と、まだ見ていない日で採点する
d = df.copy()
d['気温帯'] = pd.cut(d['temp'] * 41, bins=[0, 10, 15, 20, 25, 30, 40])
tr, te = train_test_split(d, test_size=0.2, random_state=42)
for keys in [['気温帯', 'yr'], ['mnth', 'yr', 'weathersit', 'weekday']]:
    means = tr.groupby(keys, observed=True)['cnt'].mean().rename('答え')
    pred_tr = tr.join(means, on=keys)['答え']
    pred_te = te.join(means, on=keys)['答え']
    no_ans = pred_te.isna()          # 訓練に同じ組み合わせがない日
    pred_te = pred_te.fillna(tr['cnt'].mean())
    err_tr = np.abs(tr['cnt'] - pred_tr)
    err_te = np.abs(te['cnt'] - pred_te)
    print(keys)
    print('  作った日        :', round(err_tr.mean()), '台')
    print('  まだ見ていない日:', round(err_te.mean()), '台')
    print('  答えのない日    :', int(no_ans.sum()), '日')
    print('  答えのある日だけ:', round(err_te[~no_ans].mean()), '台')


気温帯×年の段は、作った日でも見ていない日でも、外れ幅は770台ほどで変わりません。ところが、月×年×天気×曜日の細かい段は、作った日では393台なのに、まだ見ていない日では1008台も外れました。

そのわけの1つは、テスト147日のうち45日は、訓練に同じ組み合わせ（同じ月・年・天気・曜日）の日が1日もなく、答えがないことです。答えのない日は、訓練全体の平均で代用しました（`fillna`）。これが大きく外れます。答えのある102日だけで見ても、外れ幅は665台で、作った日の393台よりずっと外れます。細かく分けた段は、作った日の台数を覚えていただけで、新しい組み合わせの日には答えそのものがなかったのです。（`~no_ans`の`~`は、TrueとFalseを入れかえる記号です。）


### お手本：データを訓練とテストに分ける

- 役割：11本の列を手がかりに、データを訓練8割・テスト2割に分ける
- 入力：手がかりの表`X`と台数`y`
- 出力：`X_tr`・`X_te`・`y_tr`・`y_te`

`train_test_split`は、行をばらばらに混ぜてから分ける関数です。`test_size=0.2`でテストを2割に、`random_state=42`で混ぜ方を固定します（何度やっても同じ分け方になる）。


In [ ]:
cols = ['season', 'yr', 'mnth', 'holiday', 'weekday', 'workingday',
        'weathersit', 'temp', 'atemp', 'hum', 'windspeed']
X = df[cols].values
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                          random_state=42)
print(len(X_tr), len(X_te))


訓練が584日、テストが147日になりました。


### TODO③：訓練で学び、訓練とテストのR²を比べる

`LinearRegression`を訓練データだけで`fit`し、`score`で訓練データのR²とテストデータのR²を表示してください。


In [ ]:
# TODO: 訓練データだけで fit し、訓練の R² とテストの R² を表示する
# ヒント: model = LinearRegression().fit(X_tr, y_tr) のあと、model.score(X_tr, y_tr) と model.score(X_te, y_te)
...


訓練のR²は0.791、テストのR²は0.828です。2つがほぼ同じです（テストがたまたま少し高い）。つまり、この直線の式は作った日を覚えすぎておらず、まだ見ていない日にも同じくらい当たる、ということです。


---


## 7. 式を曲げすぎると？


6節で、直線の式は訓練とテストでほぼ同じくらい当たると分かりました。直線が暑い日の頭打ちを外すなら、式を曲げればよいのでは？ 気温の2乗、3乗……を足していくと、式は曲線になります（多項式回帰）。何乗まで使うかを次数と呼びます。曲げるほど、訓練データにはよく当たるはずです。では、テストデータには？


### お手本：3次の曲線を当てはめる

- 役割：気温だけを手がかりに、3次の曲線を訓練データで学び、訓練とテストのR²を出す
- 入力：訓練とテストの気温・台数
- 出力：2つのR²

`PolynomialFeatures(3)`は気温から2乗・3乗の列を作る部品、`Pipeline`は部品を順につなぐ道具です。


In [ ]:
x1_tr = X_tr[:, [7]]    # 8本目の列（temp）だけ
x1_te = X_te[:, [7]]
model3 = Pipeline([('poly', PolynomialFeatures(3)),
                   ('lr', LinearRegression())])
model3.fit(x1_tr, y_tr)
print(model3.score(x1_tr, y_tr), model3.score(x1_te, y_te))


In [ ]:
# @title 図：次数を変えた曲線（横軸は℃）
fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=True)
x_plot = np.linspace(x1_tr.min(), x1_tr.max(), 200).reshape(-1, 1)
for ax, deg in zip(axes, [1, 3, 8, 15]):
    m = Pipeline([('poly', PolynomialFeatures(deg)), ('lr', LinearRegression())]).fit(x1_tr, y_tr)
    ax.scatter(x1_tr * 41, y_tr, s=6, alpha=0.2, color='#0066cc')
    ax.plot(x_plot * 41, np.clip(m.predict(x_plot), -2000, 11000), color='#e63946', lw=2)
    ax.set_title(f'{deg}次'); ax.set_ylim(0, 9000); ax.set_xlabel('気温（℃）')
plt.tight_layout(); plt.show()


### TODO④：次数を変えて、訓練とテストのR²を並べる

`for`文で次数を1・3・8・15と変えながら、お手本のセルと同じことをして、次数ごとの訓練のR²とテストのR²を表示してください。


In [ ]:
# TODO: for deg in [1, 3, 8, 15]: として、次数ごとの
#       訓練 R² とテスト R² を表示する
# ヒント: お手本の PolynomialFeatures(3) の 3 を deg に変え、4つの行を for の中に入れる
...


訓練のR²は次数とともに上がっていきます（0.388 → 0.472 → 0.476 → 0.481）。テストのR²は0.404 → 0.414 → 0.408 → 0.407で、3次がいちばん高いものの、ほぼ横ばいです。訓練が584日もあると、曲線は1日1日の偶然の揺れに振り回されにくく、曲げすぎてもテストの成績はほとんど崩れません。

では、データが少なかったらどうでしょう。


### 変えてみる：訓練を20日だけにする

- 役割：訓練データを最初の20日だけにして、1・3・8次の曲線を学び、訓練とテストのR²を出す
- 入力：訓練の最初の20日（`x1_tr[:20]`・`y_tr[:20]`）と、テストの147日
- 出力：次数ごとの、訓練のR²とテストのR²

`[:20]`は、3節で見たスライスです。番号0〜19の20日を取り出します。テストは同じ147日のままです。


In [ ]:
x20 = x1_tr[:20]     # 訓練の最初の20日だけ
y20 = y_tr[:20]
for deg in [1, 3, 8]:
    m = Pipeline([('poly', PolynomialFeatures(deg)),
                  ('lr', LinearRegression())])
    m.fit(x20, y20)
    print(deg, '次:', round(m.score(x20, y20), 3),
          round(m.score(x1_te, y_te), 3))


In [ ]:
# @title 図：20日で学んだ1次と8次の曲線
x_plot = np.linspace(x1_te.min(), x1_te.max(), 400).reshape(-1, 1)
plt.scatter(x1_te * 41, y_te, s=10, alpha=0.35, color='#888888', label='テスト（147日）')
plt.scatter(x20 * 41, y20, s=45, color='#0066cc', label='訓練（20日）')
for deg, c in [(1, '#2a9d8f'), (8, '#e63946')]:
    m = Pipeline([('poly', PolynomialFeatures(deg)), ('lr', LinearRegression())]).fit(x20, y20)
    plt.plot(x_plot * 41, m.predict(x_plot), color=c, lw=2.5, label=f'{deg}次')
plt.xlim(x1_te.min() * 41, x1_te.max() * 41); plt.ylim(0, 9000)
plt.xlabel('気温（℃）'); plt.ylabel('利用台数'); plt.legend(loc='upper left')
plt.show()


訓練のR²は0.228 → 0.444 → 0.499と、曲げるほど上がります。ところがテストのR²は0.362 → −0.288 → −28.353と、マイナスに落ちました。R²のマイナスは、毎日平均と答えるより悪いということです。図を見ると、8次の曲線は青い20日に合わせて曲がり、20日の外（8℃未満や29℃超）では大きく外れています。

曲げすぎた式は、訓練データの偶然の揺れまで覚えてしまいます。これが過学習です。データが多いと目立ちませんが、少ないほどはっきり出ます。


---


## 8. 分け方の運を、ならす


6節のテストのR² 0.828は、1回の分け方で出た値です。分け方が変われば、値も変わります。そこで、データを5つに分け、1つをテストに、残りを訓練にする、を5回くり返して平均します。これを交差検証と呼びます。


### お手本：5分割の交差検証

- 役割：11本の列の直線の式を、5通りの分け方で学んでテストし、R²を5つ出す
- 入力：`X`・`y`
- 出力：5つのR²と、その平均・ばらつき


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(LinearRegression(), X, y, cv=kf,
                         scoring='r2')
print(scores.round(3))
print(scores.mean().round(3), scores.std().round(3))


5回のR²は0.72〜0.83と、分け方によって0.1ほど動きました。平均は0.783、ばらつき（標準偏差）は0.042です。1回の値だけで「実力は0.83」と言うのは、運の良い分け方を拾っていたかもしれません。実力は「平均0.78、±0.04くらい」と言うのが正直です。


---


## 9. 明日は何台？　本当の実力で答える


8節までで、式の実力をまだ見ていない日で測る方法が分かりました。最後に、運営者への答えを出します。気温と年の2つを使った直線の式を訓練データで作り、テストデータで外れ幅を測ります。明日が27℃・2年目なら、何台でしょうか。


In [ ]:
Xty = df[['temp', 'yr']].values
a_tr, a_te, b_tr, b_te = train_test_split(Xty, y, test_size=0.2,
                                          random_state=42)
model_ty = LinearRegression().fit(a_tr, b_tr)
err_tr = np.abs(b_tr - model_ty.predict(a_tr)).mean()
err_te = np.abs(b_te - model_ty.predict(a_te)).mean()
print('作った日の外れ幅  :', round(err_tr), '台')
print('まだ見ていない日  :', round(err_te), '台')
pred_t = model_ty.predict([[27 / 41, 1]])[0]     # 27℃・2年目
print('明日（27℃・2年目）:', round(pred_t), '台')


答えは「およそ6600台。まだ見ていない日での外れ幅は、平均840台くらい」です。

第1回の答え（およそ7200台）より低いのは、主に天気の違いです。第1回は25〜30℃・2年目の晴れの日（54日）だけの平均でしたが、今日の式は天気を入れていないので、晴れも雨も混ぜた見積もりになっています。

外れ幅も、同じ物差しで比べる必要があります。第1回の「500台前後」は、晴れの54日の中で、作った日で測った外れ幅でした。同じテスト147日で比べると、気温帯×年の段（6節）は773台、今日の直線は837台です。直線は境目で跳ばない代わりに、暑い日の頭打ちを外すぶん、少し外れます。

まだ言えないことも残っています。

- データを整えていない：今日のデータは、欠けた値も文字の列もない、きれいなデータでした。実務のデータはそうはいきません
- 物差しは何がよいか：R²も二乗和を使うので、大外れを重く見ている。外れ幅の平均はどの外れも同じ重さ。どちらで測るかは場面しだい
- 係数をどう読むか：年の係数2083は「2年目は同じ気温でも2083台多い」と読めるのか


---


## 目標に答えられたか


- 今日の目標は「1本の式を立て、まだ来ていない日にどれだけ当たるかを測る」ことでした。9節の答えは何台で、まだ見ていない日での外れ幅は何台でしたか？
- 4節の総当たりと5節の式一発で、谷底のa・bはどれくらい近かったでしょうか？`LinearRegression`の答えとは一致しましたか？
- TODO③で、訓練のR²とテストのR²はどれくらい違いましたか？6節の細かい段と比べて、何が違いますか？
- TODO④で、次数を上げると訓練とテストのR²はどう動きましたか？


---


## 今日の要点


- 直線の良さは、外れの二乗和で測る。二乗和を最小にするa・bを選ぶのが最小二乗法で、`LinearRegression`の中身そのもの
- `for`文で1日ずつ足した二乗和と、numpyのまとめて計算は一致する。総当たりは`for`文と`if`文（小さければ覚え直す）で書ける
- R²は、毎日平均と答える基準から、二乗和をどれだけ減らせたかの割合
- 作るのに使ったデータで測った成績は甘い。訓練とテストに分けて、まだ見ていないデータで測る
- 式を曲げすぎると、訓練には当たるがテストには当たらない（過学習）。データが少ないほど、はっきり出る
- 交差検証で、分け方の運をならす


---


## 次回へ


今日は、段を1本の式にし、まだ見ていない日で実力を測りました。ただ、今日のデータは最初からきれいに整っていました。

次回は、場面を民泊の値付けに変えます。極端な値、欠けた値、文字の列、目盛りのばらばらな列。実務のデータにつきもののこうした「汚れ」を一つずつ整えていきます。自分の部屋に、1泊いくらの値段を付ければいいでしょうか。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

6節では`random_state=42`で1回だけ分けて、テストのR²を測りました。今度は分け方を変えて、同じことを5回くり返します。

`for`文で`random_state`を0, 1, 2, 3, 4と変えながら、11本の列の`X`と`y`を訓練8割・テスト2割に分け、それぞれで`LinearRegression`を訓練データで学んで、テストのR²を5つ表示してください。5つの値は、あとで使うのでリスト`r2_list`にためておきます（`r2_list.append(値)`で1つ足せます）。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

`r2_list = []`から始め、`for rs in [0, 1, 2, 3, 4]:`の中で
`train_test_split(X, y, test_size=0.2, random_state=rs)` → `LinearRegression().fit(…)` → `.score(X_te_r, y_te_r)`の順に書きます。

</details>


### 応用②（判断）

応用①の5つのテストのR²のうち、最大と最小の差はいくつですか。丸めていない値で差を計算し、小数第2位に丸めて答えてください（例: 0.12）。


In [ ]:
# ここにコードを書く
...


<details><summary>詰まったら</summary>

リストを`np.array(r2_list)`で配列にすると、`.max()`と`.min()`が使えます。

</details>


### 応用③（解釈）

6節では、1回の分け方でテストのR²が0.828でした。この1回の値だけを根拠に「この式の実力はR²＝0.83」と報告してよいでしょうか。応用①の5つの値の幅と、8節の交差検証の結果（平均とばらつき）を根拠に、自転車シェアの運営担当者に向けて3行程度で書いてください。


（ここに3行程度で書く）


---


## 発展（任意）


### TimeSeriesSplit：日付順のデータで、未来を覗かない

8節の交差検証は、行をばらばらに混ぜてから分けました。でも、自転車シェアの台数は日付順に並んだデータです。混ぜて分けると、2012年の日で学んで2011年の日を当てる、という「未来を覗いた」採点が混ざります。

`TimeSeriesSplit`は、必ず過去で学んで未来を当てるように分けます。運営者が本当にやりたいのは、昨日までのデータで明日を当てることなので、こちらのほうが実際に近い採点です。


In [ ]:
from sklearn.model_selection import TimeSeriesSplit

tss = TimeSeriesSplit(n_splits=5)                        # 過去 → 未来の順で分割
s_kf = cross_val_score(LinearRegression(), X, y, cv=kf, scoring='r2')
s_tss = cross_val_score(LinearRegression(), X, y, cv=tss,
                        scoring='r2')
print('ランダム分割:', s_kf.round(3), ' 平均', s_kf.mean().round(3))
print('時系列分割  :', s_tss.round(3), ' 平均', s_tss.mean().round(3))


In [ ]:
# 時系列分割の各回で「どの期間で学び、どの期間を当てたか」を見る
for i, (tr_idx, te_idx) in enumerate(tss.split(X), start=1):
    print(f'{i}回目: 訓練 {df["dteday"].iloc[tr_idx[0]]}〜{df["dteday"].iloc[tr_idx[-1]]}'
          f' → テスト {df["dteday"].iloc[te_idx[0]]}〜{df["dteday"].iloc[te_idx[-1]]}'
          f'  R²={s_tss[i - 1]:.3f}')


In [ ]:
# 回ごとの外れ幅の平均（MAE）と、5回の R² の中央値
s_mae = -cross_val_score(LinearRegression(), X, y, cv=tss,
                         scoring='neg_mean_absolute_error')
print('時系列分割の外れ幅:', s_mae.round(), '台')
print('R² の中央値:', np.median(s_tss).round(3))


読み方　時系列分割のR²は、ランダム分割よりずっと低く、回によってはマイナスになります。とくに1回目は、2011年の最初の数か月だけで学んで、その先（夏）を当てています。夏も2年目の伸びも一度も見ていないので、当たらないのは当然です。

平均の −0.857は、1回目の −4.526に大きく引きずられています。5回の真ん中の値（中央値）は −0.182です。3回目（2012年1〜5月を当てる）もマイナスなのは、訓練が2012年1月3日まででほぼ2011年だけなので、2年目の伸びを知らず、台数を少なく見積もるからです。4回目は、外れ幅（約793台）は2回目（約662台）と大差ないのにR²はマイナスです。R²は、そのテスト期間の中の平均と比べる物差しなので、夏だけのように台数の振れが小さい期間では、少し外れただけでマイナスになります。回ごとの外れ幅も一緒に見ると、こうした読み違いを防げます。

ランダム分割の0.78は、「過去と未来が混ざった、少し甘い採点」だったことになります。どちらの採点が正しいかは、何に使う式かで決まります。明日を当てる式なら、時系列分割のほうが正直です。

試すなら、`n_splits`を3や10に変えて、R²がどう動くか見てください。
